In [ ]:
!pip install datasets[audio] transformers evaluate accelerate

### **模块 1：环境搭建与数据准备 (Data Preparation)**

1. 安装必要的库

---



In [3]:
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor
import numpy as np

2. 拉取音乐数据集并划分训练/测试集

In [4]:
print("正在下载 GTZAN 音乐数据集...")
gtzan = load_dataset("sanchit-gandhi/gtzan")
# 参数:
# "seed":随机数生成器定一个起点，保证每次切分出的内容一样
# "test_size":抽取10%的内容
gtzan = gtzan["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)

正在下载 GTZAN 音乐数据集...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/703 [00:00<?, ?B/s]

data/train-00000-of-00003-abaaa5719027ce(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00001-of-00003-40e2de07ad4288(…):   0%|          | 0.00/429M [00:00<?, ?B/s]

data/train-00002-of-00003-6e2eb838540a06(…):   0%|          | 0.00/436M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/999 [00:00<?, ? examples/s]

3. 提取特征提取器并获得模型的标准采样率

In [5]:
model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
  model_id, do_normalize=True, return_attention_mask=True
)
target_sampling_rate = feature_extractor.sampling_rate

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

4. 实时重采样

In [6]:
gtzan = gtzan.cast_column("audio", Audio(sampling_rate=target_sampling_rate))

5. 建立标签映射字典

In [7]:
id2label_fn = gtzan["train"].features["genre"].int2str
id2label = {str(i): id2label_fn(i) for i in range(10)}
label2id = {v: k for k, v in id2label.items()}

print("\n数据准备完成！标签映射为：", id2label)


数据准备完成！标签映射为： {'0': 'blues', '1': 'classical', '2': 'country', '3': 'disco', '4': 'hiphop', '5': 'jazz', '6': 'metal', '7': 'pop', '8': 'reggae', '9': 'rock'}


### **模块 2：特征处理与数据封装 (Feature Encoding)**


In [8]:
# 1. 定义处理函数：特征归一化 (Feature Scaling) 和长度裁剪 (截断到 30 秒)
max_duration = 30.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]] # 将 "array" 提取出来
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=target_sampling_rate,
        max_length=int(target_sampling_rate * max_duration), # 16000 * 30 限制 array长度，截取30s
        truncation=True,
        return_attention_mask=True,
    )
    return inputs


# 2. 批量处理数据集 (使用 map 函数)
print("正在对音频进行特征提取和归一化")
gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=["audio", "file"], # 扔掉原始波形和路径，节省内存
    batched=True,
    batch_size=100, # 一个batch 一起计算
    num_proc=1,
)

# 3. Trainer 要求标签列的名字必须叫 "label"
gtzan_encoded = gtzan_encoded.rename_column("genre", "label")

print("\n 处理后的数据集长这样：")
print(gtzan_encoded)

正在对音频进行特征提取和归一化 (大概需要 1-2 分钟)...


Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


特征提取完成！处理后的数据集长这样：
DatasetDict({
    train: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 899
    })
    test: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 100
    })
})


In [ ]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer
import evaluate

# 1. 加载未经微调的纯净模型，并加上分类头 (Classification Head)
# 迁移学习
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(id2label), # 加入 id2label 类别的分类头
    label2id=label2id,
    id2label=id2label,
)

# 2. 定义评估指标 (准确率 Accuracy)
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

# 3. 设置训练参数
# (注意：为了快速跑通流程，将 num_train_epochs 改为了 1)
training_args = TrainingArguments(
    output_dir="./distilhubert-finetuned-gtzan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,
    num_train_epochs=1, # 快速测试版
    warmup_steps=100,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True, # 开启混合精度，大幅加速训练并节省显存
    push_to_hub=False, # 不推送到 Hugging Face
)

# 4. 组装 Trainer 并启动
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    processing_class=feature_extractor, # 👈 【修改点】顺应最新 API，改名叫 processing_class
    compute_metrics=compute_metrics,
)

print("🚀 开始训练模型")
trainer.train()

Loading weights:   0%|          | 0/49 [00:00<?, ?it/s]

HubertForSequenceClassification LOAD REPORT from: ntu-spml/distilhubert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
projector.bias    | MISSING | 
classifier.bias   | MISSING | 
projector.weight  | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 开始训练模型！(由于设置了 Epoch=1，大约需要几分钟)...


Epoch,Training Loss,Validation Loss
